# Construção da tabela única de candidatos

## Carregamento das bases de dados

Nesta etapa são importadas as bibliotecas necessárias para manipulação dos dados e definidos os caminhos dos arquivos utilizados no processamento. O notebook utiliza três fontes principais: a base de votação dos candidatos, contendo informações eleitorais e resultados por município; a base de características municipais, contendo indicadores socioeconômicos e territoriais; e uma tabela de correspondência entre os códigos municipais utilizados pelo Tribunal Superior Eleitoral (TSE) e pelo Instituto Brasileiro de Geografia e Estatística (IBGE).

Após a definição dos arquivos de entrada e saída, as bases são carregadas em estruturas do tipo DataFrame para permitir o tratamento e integração dos dados. Também são realizadas configurações específicas de leitura, como o separador e a codificação da tabela de correspondência municipal, garantindo que os arquivos sejam interpretados corretamente. Ao final, são exibidas as quantidades iniciais de registros carregados em cada conjunto para permitir uma verificação inicial das dimensões dos dados.

In [1]:
import pandas as pd
import numpy as np

VOTING_FILE = "votacao_candidato_filtrado_limpo.csv"

MUNICIPALITY_FILE = "dados_municipios_panorama_limpo.csv"

MAPPING_FILE = "municipio_tse_ibge.csv"

OUTPUT_FILE = "candidate_representation_table.csv"

print("Loading files...")

votes = pd.read_csv(VOTING_FILE)

municipios = pd.read_csv(MUNICIPALITY_FILE)

mapping = pd.read_csv(MAPPING_FILE,sep=";",encoding="latin1")

print("Votes:", len(votes))
print("Municipalities:", len(municipios))


Loading files...
Votes: 9377797
Municipalities: 5568


## Padronização dos códigos municipais e junção das bases

Antes de integrar as bases eleitorais e municipais, é necessário garantir que os identificadores utilizados como chave de ligação estejam no mesmo formato. Nesta etapa, os códigos municipais presentes na base de votação, na tabela de correspondência TSE-IBGE e na base municipal são convertidos para o formato numérico, eliminando possíveis inconsistências causadas por valores armazenados como texto.

Em seguida, é realizada a conversão dos códigos municipais do padrão utilizado pelo TSE para o padrão utilizado pelo IBGE. Essa etapa é necessária porque as duas bases principais utilizam identificadores diferentes para representar os municípios. Após o mapeamento, registros que não possuem correspondência válida são removidos.

Com os códigos padronizados, a base eleitoral é integrada à base municipal por meio do identificador IBGE. O resultado é uma única tabela contendo, para cada registro eleitoral, tanto as informações do candidato quanto as características do município onde os votos foram obtidos.

In [2]:
votes["Código município"] = pd.to_numeric(votes["Código município"],errors="coerce")

mapping["CD_MUNICIPIO_TSE"] = pd.to_numeric(mapping["CD_MUNICIPIO_TSE"],errors="coerce")

mapping["CD_MUNICIPIO_IBGE"] = pd.to_numeric(mapping["CD_MUNICIPIO_IBGE"],errors="coerce")

municipios["municipio_id"] = pd.to_numeric(municipios["municipio_id"],errors="coerce")

print("Mapping municipalities...")

votes = votes.merge(
    mapping[["CD_MUNICIPIO_TSE","CD_MUNICIPIO_IBGE"]],
    left_on="Código município",
    right_on="CD_MUNICIPIO_TSE",
    how="left"
)

votes = votes[votes["CD_MUNICIPIO_IBGE"].notna()].copy()

print("Rows after mapping:",len(votes))

print("Joining municipality data...")

df = votes.merge(municipios,left_on="CD_MUNICIPIO_IBGE",right_on="municipio_id",how="inner")

print("Rows after join:",len(df))

Mapping municipalities...
Rows after mapping: 9375444
Joining municipality data...
Rows after join: 9374385


## Limpeza dos dados e seleção das variáveis

Após a integração das bases, é realizada uma etapa de limpeza para garantir que as variáveis utilizadas estejam em formatos adequados. As informações relacionadas à participação eleitoral, votos válidos e votos nominais são convertidas para valores numéricos, permitindo operações matemáticas e agregações posteriores.

Registros sem informação válida de participação eleitoral são removidos, assim como observações com participação igual a zero, pois não contribuem para a análise da representação eleitoral dos candidatos.

Em seguida, são definidas as variáveis que identificam unicamente cada candidato, considerando cargo, nome, partido e turno da eleição. Também são selecionadas as características individuais dos candidatos, como gênero, raça, escolaridade, ocupação e situação eleitoral, que serão mantidas na tabela final.

Além disso, são identificadas automaticamente as variáveis municipais numéricas e categóricas. Essa separação permite aplicar diferentes estratégias de agregação posteriormente: variáveis numéricas serão combinadas por médias ponderadas, enquanto variáveis categóricas serão transformadas em proporções representativas do contexto municipal.

In [3]:
df["vote_share"] = pd.to_numeric(df["vote_share"],errors="coerce")

df["Votos válidos"] = pd.to_numeric(df["Votos válidos"],errors="coerce")

df["Votos nominais"] = pd.to_numeric(df["Votos nominais"],errors="coerce")

df = df[df["vote_share"].notna()].copy()

df = df[df["vote_share"] > 0].copy()

print("Rows with vote_share > 0:",len(df))

candidate_cols = [
    "Cargo",
    "Nome candidato",
    "Partido",
    "Turno"
]

candidate_info_cols = [
    "Cor/raça",
    "Estado civil",
    "Faixa etária",
    "Gênero",
    "Grau de instrução",
    "Ocupação",
    "Situação totalização",
    "UF",
    "Região"
]

SUCCESS_LABELS = [
    "Eleito",
    "Suplente"
]

exclude_cols = set(
    candidate_cols
    + candidate_info_cols
    + [
        "Código município",
        "CD_MUNICIPIO_TSE",
        "CD_MUNICIPIO_IBGE",
        "municipio",
        "municipio_id",
        "Município",
        "Número candidato",
        "vote_share",
        "Votos válidos",
        "Votos nominais",
        "Zona"
    ]
)

numeric_cols = []

for col in municipios.columns:

    if col in exclude_cols:
        continue

    if pd.api.types.is_numeric_dtype(municipios[col]):
        numeric_cols.append(col)

categorical_cols = []

for col in municipios.columns:

    if col in exclude_cols:
        continue

    if municipios[col].dtype == "object":
        categorical_cols.append(col)

print("\nNumeric municipality columns:")
print(len(numeric_cols))

print("\nCategorical municipality columns:")
print(categorical_cols)

Rows with vote_share > 0: 2879689

Numeric municipality columns:
27

Categorical municipality columns:
[]


## Construção da tabela de representação dos candidatos

Nesta etapa é construída uma nova tabela em que cada linha representa um candidato e suas características eleitorais e municipais agregadas. Para isso, os registros são agrupados utilizando as variáveis de identificação do candidato: cargo, nome, partido e turno.

Como um mesmo candidato pode receber votos em diversos municípios, é necessário resumir essas informações em uma única representação. Para isso, é utilizado um peso baseado na combinação entre a participação de votos do candidato no município (`vote_share`) e o tamanho do eleitorado local, calculado utilizando o logaritmo do número de votos válidos. Dessa forma, municípios onde o candidato teve maior relevância eleitoral possuem maior influência na representação final.

São calculadas métricas eleitorais agregadas, como participação total de votos, média e máximo de participação municipal, quantidade total de votos nominais, quantidade total de votos válidos e número de municípios onde o candidato recebeu votos.

As variáveis municipais numéricas são agregadas utilizando médias ponderadas pelos pesos eleitorais. Já as variáveis categóricas são transformadas em proporções ponderadas, criando novas variáveis que representam a distribuição das características municipais associadas ao candidato. Por exemplo, uma categoria municipal passa a ser representada pela proporção de municípios onde ela aparece, considerando a importância eleitoral de cada localidade.

Também é criada a variável objetivo `success`, que indica se o candidato obteve sucesso eleitoral. Essa variável recebe valor 1 quando a situação final do candidato corresponde a eleito ou suplente, e 0 caso contrário.

In [4]:
candidate_rows = []

groups = df.groupby(candidate_cols)

total_groups = len(groups)

print("\nCandidates:", total_groups)

for idx, (candidate_key, group) in enumerate(groups):

    if idx % 1000 == 0:
        print(idx, "/", total_groups)

    weights = (group["vote_share"].values * np.log1p(group["Votos válidos"].values))

    if weights.sum() == 0:
        continue

    row = {}

    for c, v in zip(candidate_cols,candidate_key):
        row[c] = v


    for col in candidate_info_cols:

        if col in group.columns:

            row[col] = (group[col].iloc[0])

    status = row["Situação totalização"]

    row["success"] = int(status in SUCCESS_LABELS)

    row["total_vote_share"] = (group["vote_share"].sum())

    row["mean_vote_share"] = (group["vote_share"].mean())

    row["max_vote_share"] = (group["vote_share"].max())

    row["total_nominal_votes"] = (group["Votos nominais"].sum())

    row["total_valid_votes"] = (group["Votos válidos"].sum())

    row["n_municipalities"] = (len(group))

    for col in numeric_cols:

        vals = pd.to_numeric(group[col],errors="coerce")

        mask = vals.notna()

        if mask.sum() == 0:
            row[col] = np.nan
            continue

        row[col] = np.average(vals[mask],weights=weights[mask])

    for col in categorical_cols:

        temp = pd.DataFrame({
            "cat": group[col],
            "w": weights
        })

        shares = (temp.groupby("cat")["w"].sum() / weights.sum())

        for category, share in shares.items():

            safe_name = (
                str(category)
                .replace(" ", "_")
                .replace("/", "_")
                .replace("-", "_")
                .replace("(", "")
                .replace(")", "")
            )

            row[f"{col}_{safe_name}"] = share

    candidate_rows.append(row)


Candidates: 25186
0 / 25186
1000 / 25186
2000 / 25186
3000 / 25186
4000 / 25186
5000 / 25186
6000 / 25186
7000 / 25186
8000 / 25186
9000 / 25186
10000 / 25186
11000 / 25186
12000 / 25186
13000 / 25186
14000 / 25186
15000 / 25186
16000 / 25186
17000 / 25186
18000 / 25186
19000 / 25186
20000 / 25186
21000 / 25186
22000 / 25186
23000 / 25186
24000 / 25186
25000 / 25186


## Geração da tabela final e exportação

Após a construção das representações individuais dos candidatos, os registros gerados são convertidos em uma nova tabela estruturada. Valores ausentes resultantes das agregações são substituídos por zero para garantir que o conjunto final esteja completo e pronto para análises posteriores.

Nesta etapa também são realizadas verificações finais sobre o tamanho da tabela gerada e a distribuição da variável de sucesso eleitoral. Essas verificações permitem avaliar quantos candidatos foram representados e qual a proporção de candidatos classificados como bem-sucedidos.

Por fim, a tabela final é salva em um novo arquivo CSV contendo uma linha por candidato, suas características individuais, métricas eleitorais e a representação agregada dos municípios onde recebeu votos. Esse arquivo constitui a base preparada para as etapas seguintes de modelagem ou análise exploratória.

In [5]:
candidate_table = pd.DataFrame(candidate_rows)

candidate_table.fillna(0,inplace=True)

print("\nFinal shape:")
print(candidate_table.shape)

print("\nSuccess distribution:")

print(candidate_table["success"].value_counts())

print("\nSuccess rate:",round(100 *candidate_table["success"].mean(),2),"%")

candidate_table.to_csv(OUTPUT_FILE,index=False)

print("\nSaved:",OUTPUT_FILE)

print("\nDone.")


Final shape:
(25186, 47)

Success distribution:
success
1    16237
0     8949
Name: count, dtype: int64

Success rate: 64.47 %

Saved: candidate_representation_table.csv

Done.
